<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Check what a vision-language model says

**[Check what a vision-language model says](https://learning.nextia-ai.com/courses/vision/m07/check-what-a-vision-language-model-says/)** · Vision AI: See, Count and Inspect · Module 7, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score the recorded answers of two vision-language models against labels, look at the wrong answers, put an honest error bar on each accuracy, and decide which product attributes can be used without a person's review.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account, no key and no model: every answer was recorded once and is replayed from a file. |
| **You should know** | The recorded answers and structured output, from [Vision-language models](https://learning.nextia-ai.com/courses/vision/m07/vision-language-models/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m07/check-what-a-vision-language-model-says/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M07-L03-check-what-a-vision-language-model-says/check-what-a-vision-language-model-says-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 21 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "archives/pallets.zip": "cec4b211fa4e434936fc0c1980accfc74392f1ced9de6c04b5421b9ec6af56e3",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The answers and the labels

The 50 attribute labels in `data/recordings/attribute_labels.csv` were made by **one person, by eye**. Where two answers are fair, the label lists both with `|` (for example `yellow|orange`). Where the photo does not show the material, the label is `unclear`, and that field is not scored. The product type comes from the catalogue labels.

In [ ]:
import json
catalogue = pd.read_csv("data/catalogue/manifest.csv").set_index("image_id")
labels = pd.read_csv("data/recordings/attribute_labels.csv").set_index("image_id")
hosted = [json.loads(line) for line in open("data/recordings/vlm-hosted.jsonl")]
local = [json.loads(line) for line in open("data/recordings/vlm-local.jsonl")]
attrs = {"hosted": [a for a in hosted if a["task"] == "attributes"],
         "local": [a for a in local if a["task"] == "attributes"]}
rows = [{"id": a["id"], **{f"said_{f}": a["parsed"][f] for f in ("product_type", "main_colour", "material")},
         **{f"label_{f}": labels.loc[a["id"], f] for f in ("product_type", "main_colour", "material")}}
        for a in attrs["hosted"]]
print(pd.DataFrame(rows).head(8).to_string(index=False))

## 2. Score each field

> **Your turn.** Write `accuracy(answers)`. For each of the three fields, it returns the share of answers whose value is one of the label's alternatives (`label.split("|")`). Skip a field whose label is `unclear`. Compare in lower case, with spaces and `-` turned into `_`, as `norm` does. Round each share to 3 decimals, and return a dictionary from field to share. Then put `accuracy(attrs["hosted"])` in `hosted_accuracy`.

In [ ]:
FIELDS = ("product_type", "main_colour", "material")

def norm(v):
    return str(v or "").strip().lower().replace(" ", "_").replace("-", "_")

def accuracy(answers):
    out = {}
    for f in FIELDS:
        right = scored = 0
        for a in answers:
            truth = labels.loc[a["id"], f].split("|")
            if truth == ["unclear"]:
                continue
            scored += 1
            right += norm((a["parsed"] or {}).get(f)) in [norm(t) for t in truth]
        out[f] = round(right / scored, 3)
    return out

hosted_accuracy = accuracy(attrs["hosted"])
print(hosted_accuracy)

In [ ]:
expect_hash('hosted_accuracy', hosted_accuracy, '52deb87895416e7f')

Now change one condition: score the small local model with the same function. The recording script read its broken answers as well as it could (the last lesson), so its scores are as good as they can be.

In [ ]:
local_accuracy = accuracy(attrs["local"]) if hosted_accuracy is not ... else None
print(local_accuracy)

> **Check.** The local model: {'product_type': 0.2, 'main_colour': 0.4, 'material': 0.25}. The hosted model is far better, but it is not perfect on any field.

## 3. Look at every wrong answer

A number says how often. Only the photos say **why**. Look at each wrong product type, with the photo:

In [ ]:
wrong = [a for a in attrs["hosted"] if a["parsed"]["product_type"] != labels.loc[a["id"], "product_type"]]
show(["data/" + catalogue.loc[a["id"], "file"] for a in wrong],
     [f"label {labels.loc[a['id'], 'product_type']}\nsaid {a['parsed']['product_type']}" for a in wrong], size=2)
for a in wrong:
    print(a["id"], "|", a["parsed"]["alt_text"])

> **Predict, then decide.** For each photo, is the model wrong, is the label wrong, or are both fair? Write your answer for each one before you read the lesson. Some catalogue "lamps" are a light bulb and a glass furnace: the label is not always the truth.

## 4. How sure is 84%?

The hosted model got 41 of 44 materials right: 93.2%. With only 44 photos, the true rate on all of Larkfield's photos could be quite different. A **95% confidence interval** gives a range of rates that agree with what we saw. The Wilson interval is a common one for a share:

In [ ]:
import math

def wilson(right, scored, z=1.96):
    p = right / scored
    d = 1 + z * z / scored
    centre = (p + z * z / (2 * scored)) / d
    half = z * math.sqrt(p * (1 - p) / scored + z * z / (4 * scored * scored)) / d
    return round(centre - half, 3), round(centre + half, 3)

for f, (right, scored) in {"product_type": (42, 50), "main_colour": (43, 50), "material": (41, 44)}.items():
    low, high = wilson(right, scored)
    print(f"{f:12s} {right}/{scored} = {right / scored:.3f}   95% interval {low:.3f} to {high:.3f}")

> **Check.** For material, the interval is 0.818 to 0.977. If Omar needs 95% before a field goes on the website without review, 44 photos cannot show it: the interval reaches far below 95%. He needs more labelled photos, or a person checks this field.

## 5. Counting boxes

Both models also counted the boxes on the 30 drawn pallet test photos. These are clean drawings, made by our code: real pallets in a real warehouse are harder.

In [ ]:
pallets = json.load(open("data/pallets/test.json"))
truth = {im["file_name"].split("/")[-1][:-4]: im for im in pallets["images"]}
for name, answers in (("hosted", hosted), ("local", local)):
    counts = [(truth[a["id"]]["count"], (a["parsed"] or {}).get("boxes") or 0, truth[a["id"]]["crowding"])
              for a in answers if a["task"] == "count"]
    errors = pd.DataFrame(counts, columns=["true", "said", "crowding"])
    errors["error"] = (errors["said"] - errors["true"]).abs()
    print(name, errors.groupby("crowding")["error"].mean().round(1).to_dict(), "all:", round(errors["error"].mean(), 1))

## 6. What 1,000 photos cost

The recordings keep the price of each hosted answer, from the tokens it used.

In [ ]:
usd = sum(a["usd"] for a in hosted)
print(f"{len(hosted)} hosted answers: {usd:.4f} dollars, {usd / len(hosted) * 1000:.2f} dollars per 1,000 photos")
print(f"mean time: hosted {np.mean([a['seconds'] for a in hosted]):.1f} s, local {np.mean([a['seconds'] for a in local]):.1f} s an answer")

> **Check.** About 1.38 dollars per 1,000 photos for the hosted model, at the price when the answers were recorded. Prices change: check the provider's current price page. The local model costs no money per answer, but about 4 seconds of CPU each.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m07/check-what-a-vision-language-model-says/) has the marked answers, the comparison with CLIP and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [When not to build it](https://learning.nextia-ai.com/courses/vision/m07/when-not-to-build-it/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m07/check-what-a-vision-language-model-says/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.